In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")

# Encode a sentence
text = "I love building AI systems."
inputs = tokenizer(text, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[ 101, 1045, 2293, 2311, 9932, 3001, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [3]:
# Get the actual embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Last hidden state = embeddings for each token
embeddings = outputs.last_hidden_state
print(f"Shape: {embeddings.shape}")

Shape: torch.Size([1, 8, 768])


In [4]:
# What does one token's embedding actually look like?
token_zero = embeddings[0][0]  # first sentence, first token ([CLS])
print(f"Token: [CLS]")
print(f"Embedding length: {token_zero.shape}")
print(f"First 10 values: {token_zero[:10]}")

Token: [CLS]
Embedding length: torch.Size([768])
First 10 values: tensor([ 0.0570,  0.0388, -0.2291, -0.2583, -0.1511, -0.1005,  0.2425,  0.5756,
        -0.1704, -0.3696])


In [5]:
# Do similar words have similar embeddings?
from torch.nn.functional import cosine_similarity

words = ["king", "queen", "man", "woman", "python", "javascript"]

# Get embedding for each word individually
def get_embedding(word):
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)
    return output.last_hidden_state[0][1]  # index 1 skips [CLS]

embeddings_dict = {word: get_embedding(word) for word in words}

# Compare cosine similarity between pairs
pairs = [
    ("king", "queen"),
    ("king", "man"),
    ("man", "woman"),
    ("king", "python"),
    ("python", "javascript"),
]

for a, b in pairs:
    sim = cosine_similarity(embeddings_dict[a].unsqueeze(0), 
                           embeddings_dict[b].unsqueeze(0))
    print(f"{a:12} vs {b:12} → {sim.item():.4f}")

king         vs queen        → 0.8600
king         vs man          → 0.7857
man          vs woman        → 0.8729
king         vs python       → 0.5233
python       vs javascript   → 0.6398


In [6]:
# Sentence embeddings — mean pooling
def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)
    # Average all token embeddings = sentence embedding
    return output.last_hidden_state.mean(dim=1).squeeze()

sentences = [
    "I love building AI systems.",
    "I enjoy creating artificial intelligence applications.",
    "The weather in Sydney is beautiful today.",
    "Python is a great programming language.",
]

embs = {s: get_sentence_embedding(s) for s in sentences}

# Compare all pairs
for i, a in enumerate(sentences):
    for b in sentences[i+1:]:
        sim = cosine_similarity(embs[a].unsqueeze(0), embs[b].unsqueeze(0))
        print(f"{sim.item():.4f} | {a[:40]} vs {b[:40]}")

0.9270 | I love building AI systems. vs I enjoy creating artificial intelligence
0.6908 | I love building AI systems. vs The weather in Sydney is beautiful today
0.7454 | I love building AI systems. vs Python is a great programming language.
0.6759 | I enjoy creating artificial intelligence vs The weather in Sydney is beautiful today
0.7760 | I enjoy creating artificial intelligence vs Python is a great programming language.
0.6397 | The weather in Sydney is beautiful today vs Python is a great programming language.
